# Experimental test 1 Result

* vllm기준으로 accuracy와 f1 score를 테스트
* few-shot테스트를 위해서 sc.Self_Consistency의 두번째 파라미터를 1, 2, 3, 4 로 변경하여 실험 수행
* 이외의 파라미터는 고정 
    * fewshot 테스트에 활용할 질문의 개수  : 60
    * 소스코드 포함여부  : 'Y'           
    * 반복횟수 : 5회                
    * 시스템프롬프트 'sys_prompt10'
    * self-consistency 횟수 : 5
    * temperature : 0.01
    * 엑셀버전 : 'ver7'
* 이후 결과에 대해서 스코어 비교 진행 


In [4]:
import sys, os
sys.path.insert(1, '..')
sys.path.insert(1, '../..')
import pandas as pd
from config import config as conf
import re
import numpy as np
from sklearn import metrics



In [ ]:
def sc_calc_acc_condition_with_temp_with_sc(llm_model, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'{conf.DATA_PATH}/{conf.ANNO_RESULT}'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')]

    df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp['o_result'] = tmp['result'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp = tmp[tmp['o_result'].isin(['1', '0', '2'])]

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list


In [6]:
    # process3 = Process(target=task, args=('v',              # llm_model
    #                                       4,                # few_shot_n
    #                                       60,                # test_n(# of question for test)
    #                                       'Y',              # q_src_yn 
    #                                       5,                # iteration num
    #                                       'sys_prompt10',   # prompt ver
    #                                       5,                # self-consistency number
    #                                       0.01,             # temperature
    #                                       'ver7'            # excel_verion
    #                                       ))

In [7]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('v', 1, 60, 'Y', 5, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

              precision    recall  f1-score   support

           0      0.483     0.824     0.609        34
           1      0.882     0.694     0.777       108
           2      0.889     0.857     0.873        28

    accuracy                          0.747       170
   macro avg      0.751     0.792     0.753       170
weighted avg      0.804     0.747     0.759       170

v_result_1_60_Y :  74.70588235294117
[np.float64(75.0), np.float64(63.63636363636363), np.float64(79.41176470588235), np.float64(78.125), np.float64(77.41935483870968)]


In [8]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('v', 2, 60, 'Y', 5, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

              precision    recall  f1-score   support

           0      0.789     0.776     0.783        58
           1      0.819     0.810     0.814        84
           2      0.862     0.926     0.893        27

    accuracy                          0.817       169
   macro avg      0.824     0.837     0.830       169
weighted avg      0.816     0.817     0.816       169

v_result_2_60_Y :  81.65680473372781
[np.float64(83.33333333333334), np.float64(75.67567567567568), np.float64(77.5), np.float64(86.20689655172413), np.float64(87.87878787878788)]


In [9]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('v', 3, 60, 'Y', 5, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

              precision    recall  f1-score   support

           0      0.932     0.840     0.883        81
           1      0.704     0.844     0.768        45
           2      0.923     0.889     0.906        27

    accuracy                          0.850       153
   macro avg      0.853     0.858     0.852       153
weighted avg      0.863     0.850     0.853       153

v_result_3_60_Y :  84.9673202614379
[np.float64(86.20689655172413), np.float64(87.5), np.float64(81.25), np.float64(86.66666666666667), np.float64(83.33333333333334)]


In [10]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('v', 4, 60, 'Y', 5, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

              precision    recall  f1-score   support

           0      1.000     0.783     0.878        92
           1      0.627     0.925     0.747        40
           2      0.870     0.909     0.889        22

    accuracy                          0.838       154
   macro avg      0.832     0.872     0.838       154
weighted avg      0.885     0.838     0.846       154

v_result_4_60_Y :  83.76623376623377
[np.float64(78.125), np.float64(80.64516129032258), np.float64(83.33333333333334), np.float64(88.88888888888889), np.float64(88.23529411764706)]
